In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import json
import copy

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset_full = torchvision.datasets.EMNIST(root='./data', split="balanced", train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.EMNIST(root='./data', split="balanced", train=False, download=True, transform=transform)

train_size = int(0.8 * len(train_dataset_full))
val_size = len(train_dataset_full) - train_size

train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size], generator=torch.Generator().manual_seed(SEED))

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")
x_batch, y_batch = next(iter(train_loader))
print(f"Batch shape: {x_batch.shape}, Labels shape: {y_batch.shape}")
print(f"Value range: [{x_batch.min():.2f}, {x_batch.max():.2f}]")
print(f"Number of classes: {len(torch.unique(y_batch))}")

class MLP(nn.Module):
    def __init__(self, input_size=28*28, hidden_sizes=[512, 256], num_classes=47, 
                 dropout_p=0.0, use_bn=False):
        super(MLP, self).__init__()
        layers = []
        prev_size = input_size
        
        for h_size in hidden_sizes:
            layers.append(nn.Linear(prev_size, h_size))
            if use_bn:
                layers.append(nn.BatchNorm1d(h_size))
            layers.append(nn.ReLU())
            if dropout_p > 0:
                layers.append(nn.Dropout(p=dropout_p))
            prev_size = h_size
            
        layers.append(nn.Linear(prev_size, num_classes))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.network(x)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * x.size(0)
        _, predicted = torch.max(logits, 1)
        total += y.size(0)
        correct += (predicted == y).sum().item()
        
    return total_loss / total, 100 * correct / total

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            
            total_loss += loss.item() * x.size(0)
            _, predicted = torch.max(logits, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()
            
    return total_loss / total, 100 * correct / total

os.makedirs("artifacts/figures", exist_ok=True)

results = []
best_val_acc = -1
best_model_state = None
best_config = None
best_experiment_id = ""

def run_experiment(exp_id, config, epochs, patience=None):
    global best_val_acc, best_model_state, best_config, best_experiment_id
    
    model = MLP(
        hidden_sizes=config['hidden_sizes'], 
        dropout_p=config['dropout'], 
        use_bn=config['use_bn'],
        num_classes=47
    ).to(device)
    
    if config['optimizer'] == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
    else:
        optimizer = optim.SGD(model.parameters(), lr=config['lr'], momentum=config['momentum'], weight_decay=config['weight_decay'])
        
    criterion = nn.CrossEntropyLoss()
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_epoch_val_acc = -1
    patience_counter = 0
    actual_epochs_trained = 0
    
    print(f"\n--- Running {exp_id} ---")
    
    for epoch in range(epochs):
        t_loss, t_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        v_loss, v_acc = evaluate(model, val_loader, criterion, device)
        
        history['train_loss'].append(t_loss)
        history['val_loss'].append(v_loss)
        history['train_acc'].append(t_acc)
        history['val_acc'].append(v_acc)
        actual_epochs_trained = epoch + 1
        
        print(f"Epoch {epoch+1}: Train Loss {t_loss:.4f}, Val Acc {v_acc:.2f}%")
        
        if patience is not None:
            if v_acc > best_epoch_val_acc:
                best_epoch_val_acc = v_acc
                patience_counter = 0
            else:
                patience_counter += 1
            
            if patience_counter >= patience:
                print(f"EarlyStopping triggered at epoch {epoch+1}")
                break
    
    final_val_acc = history['val_acc'][-1]
    best_val_acc_in_run = max(history['val_acc'])
    
    results.append({
        'experiment_id': exp_id,
        'dataset': 'EMNIST',
        'seed': SEED,
        'model_summary': f"Hidden:{config['hidden_sizes']}, DO:{config['dropout']}, BN:{config['use_bn']}",
        'optimizer': config['optimizer'],
        'lr': config['lr'],
        'momentum': config['momentum'],
        'weight_decay': config['weight_decay'],
        'epochs_trained': actual_epochs_trained,
        'best_val_accuracy': best_val_acc_in_run,
        'best_val_loss': min(history['val_loss'])
    })
    
    return history, model

BASE_HIDDEN = [512, 256]

cfg_e1 = {'hidden_sizes': BASE_HIDDEN, 'dropout': 0.0, 'use_bn': False, 'optimizer': 'Adam', 'lr': 1e-3, 'momentum': 0, 'weight_decay': 0}
h_e1, m_e1 = run_experiment('E1', cfg_e1, epochs=20)

cfg_e2 = {'hidden_sizes': BASE_HIDDEN, 'dropout': 0.3, 'use_bn': False, 'optimizer': 'Adam', 'lr': 1e-3, 'momentum': 0, 'weight_decay': 0}
h_e2, m_e2 = run_experiment('E2', cfg_e2, epochs=20)

cfg_e3 = {'hidden_sizes': BASE_HIDDEN, 'dropout': 0.0, 'use_bn': True, 'optimizer': 'Adam', 'lr': 1e-3, 'momentum': 0, 'weight_decay': 0}
h_e3, m_e3 = run_experiment('E3', cfg_e3, epochs=20)

res_e2 = results[-2]
res_e3 = results[-1]

best_arch_cfg = cfg_e2 if res_e2['best_val_accuracy'] > res_e3['best_val_accuracy'] else cfg_e3
best_arch_name = "E2-Dropout" if res_e2['best_val_accuracy'] > res_e3['best_val_accuracy'] else "E3-BN"
print(f"\nSelected architecture for E4 based on val_acc: {best_arch_name}")

cfg_e4 = best_arch_cfg.copy()
model_e4 = MLP(hidden_sizes=cfg_e4['hidden_sizes'], dropout_p=cfg_e4['dropout'], use_bn=cfg_e4['use_bn'], num_classes=47).to(device)
optimizer_e4 = optim.Adam(model_e4.parameters(), lr=cfg_e4['lr'])
criterion = nn.CrossEntropyLoss()

h_e4 = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
patience = 5
counter = 0
best_val_acc_e4 = -1
best_state_e4 = None
epochs_e4 = 0

print(f"\n--- Running E4 (EarlyStopping on {best_arch_name}) ---")
for epoch in range(30):
    t_loss, t_acc = train_one_epoch(model_e4, train_loader, optimizer_e4, criterion, device)
    v_loss, v_acc = evaluate(model_e4, val_loader, criterion, device)
    
    h_e4['train_loss'].append(t_loss)
    h_e4['val_loss'].append(v_loss)
    h_e4['train_acc'].append(t_acc)
    h_e4['val_acc'].append(v_acc)
    epochs_e4 = epoch + 1
    
    if v_acc > best_val_acc_e4:
        best_val_acc_e4 = v_acc
        best_state_e4 = copy.deepcopy(model_e4.state_dict())
        counter = 0
    else:
        counter += 1
        
    if counter >= patience:
        print(f"EarlyStopping at epoch {epoch+1}")
        break

results.append({
    'experiment_id': 'E4',
    'dataset': 'EMNIST', 'seed': SEED,
    'model_summary': f"Based on {best_arch_name}",
    'optimizer': 'Adam', 'lr': cfg_e4['lr'], 'momentum': 0, 'weight_decay': 0,
    'epochs_trained': epochs_e4,
    'best_val_accuracy': best_val_acc_e4,
    'best_val_loss': min(h_e4['val_loss'])
})

torch.save(best_state_e4, "artifacts/best_model.pt")
best_config_data = {
    "experiment": "E4",
    "architecture": best_arch_cfg,
    "seed": SEED,
    "dataset": "EMNIST",
    "num_classes": 47,
    "best_val_accuracy": best_val_acc_e4
}
with open("artifacts/best_config.json", "w") as f:
    json.dump(best_config_data, f, indent=2)

cfg_o1 = best_arch_cfg.copy()
cfg_o1['lr'] = 1e-1
h_o1, _ = run_experiment('O1', cfg_o1, epochs=8)

cfg_o2 = best_arch_cfg.copy()
cfg_o2['lr'] = 1e-5
h_o2, _ = run_experiment('O2', cfg_o2, epochs=8)

cfg_o3 = best_arch_cfg.copy()
cfg_o3['optimizer'] = 'SGD'
cfg_o3['lr'] = 1e-2
cfg_o3['momentum'] = 0.9
cfg_o3['weight_decay'] = 1e-4
h_o3, _ = run_experiment('O3', cfg_o3, epochs=15)

plt.style.use('seaborn-v0_8')

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(h_e4['train_loss'], label='Train Loss')
plt.plot(h_e4['val_loss'], label='Val Loss')
plt.title(f"E4 Loss (Best Model)")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(h_e4['train_acc'], label='Train Acc')
plt.plot(h_e4['val_acc'], label='Val Acc')
plt.title(f"E4 Accuracy (Best: {best_val_acc_e4:.2f}%)")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.savefig("artifacts/figures/curves_best.png")
plt.close()

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(h_o1['val_loss'], 'r-', label='O1: LR=1e-1 (High)', linewidth=2)
plt.title("O1: Too High LR (Divergence/Instability)")
plt.xlabel("Epoch")
plt.ylabel("Val Loss")
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(h_o2['val_loss'], 'b-', label='O2: LR=1e-5 (Low)', linewidth=2)
plt.title("O2: Too Low LR (No Learning)")
plt.xlabel("Epoch")
plt.ylabel("Val Loss")
plt.grid(True)

plt.tight_layout()
plt.savefig("artifacts/figures/curves_lr_extremes.png")
plt.close()

df_results = pd.DataFrame(results)
df_results.to_csv("artifacts/runs.csv", index=False)
print("Saved artifacts/runs.csv")

model_final = MLP(hidden_sizes=best_arch_cfg['hidden_sizes'], dropout_p=best_arch_cfg['dropout'], use_bn=best_arch_cfg['use_bn'], num_classes=47).to(device)
model_final.load_state_dict(best_state_e4)
test_loss, test_acc = evaluate(model_final, test_loader, criterion, device)
print(f"\n=== FINAL TEST ACCURACY (Best Model E4): {test_acc:.2f}% ===")

Using device: cpu


100%|████████████████████████████████████████| 562M/562M [01:44<00:00, 5.39MB/s]


Train size: 90240, Val size: 22560, Test size: 18800
Batch shape: torch.Size([128, 1, 28, 28]), Labels shape: torch.Size([128])
Value range: [-1.00, 1.00]
Number of classes: 45

--- Running E1 ---
Epoch 1: Train Loss 1.1392, Val Acc 76.86%
Epoch 2: Train Loss 0.6347, Val Acc 80.73%
Epoch 3: Train Loss 0.5260, Val Acc 81.73%
Epoch 4: Train Loss 0.4644, Val Acc 82.39%
Epoch 5: Train Loss 0.4226, Val Acc 83.45%
Epoch 6: Train Loss 0.3930, Val Acc 83.44%
Epoch 7: Train Loss 0.3654, Val Acc 83.75%
Epoch 8: Train Loss 0.3476, Val Acc 83.53%
Epoch 9: Train Loss 0.3250, Val Acc 83.98%
Epoch 10: Train Loss 0.3060, Val Acc 83.70%
Epoch 11: Train Loss 0.2928, Val Acc 83.99%
Epoch 12: Train Loss 0.2754, Val Acc 83.78%
Epoch 13: Train Loss 0.2627, Val Acc 83.57%
Epoch 14: Train Loss 0.2516, Val Acc 83.64%
Epoch 15: Train Loss 0.2414, Val Acc 83.82%
Epoch 16: Train Loss 0.2290, Val Acc 83.64%
Epoch 17: Train Loss 0.2205, Val Acc 83.23%
Epoch 18: Train Loss 0.2149, Val Acc 83.11%
Epoch 19: Train Loss

In [6]:
import torch
import torchvision
import sys

print(f"Python: {sys.version.split()[0]}")
print(f"torch: {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

Python: 3.12.12
torch: 2.10.0
torchvision: 0.25.0
Device: CPU
